# GraphRAG — Evaluation Notebook
**Corpus:** Attention is All You Need (Vaswani et al., 2017)

This notebook:
1. Generates 125 evaluation questions about the Transformer paper
2. Measures router accuracy on 30 manually labeled queries
3. Runs all 3 systems on all questions
4. Runs LLM-as-judge pairwise comparisons
5. Computes win rates — ready for the report

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, json, time
PROJECT_ROOT = '/content/drive/MyDrive/GraphRAG'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

GROQ_API_KEY = 'gsk_paste_your_key_here'
if GROQ_API_KEY == 'gsk_paste_your_key_here':
    raise ValueError('Paste your Groq key first')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.makedirs('evaluation', exist_ok=True)

from src.config import cfg
cfg['llm']['generation_provider'] = 'groq'
cfg['llm']['generation_model']    = 'llama-3.3-70b-versatile'

print('✅ Cell 1 passed — setup complete')

In [ ]:
# ── Cell 2: Initialize all engines ───────────────────────────
from src.llm_client import LLMClient
from src.vector_store import VectorStore
from src.router import QueryRouter
from src.community_detection import CommunityDetector
from src.community_summarizer import CommunitySummarizer
from src.graph_query import GraphQueryEngine
from src.graph_traversal import GraphTraversal
from src.query_engine import QueryEngine
from src.config import GRAPH_PATH
from src.chunker import load_all_chunks

print('Initializing engines...')

llm    = LLMClient(purpose='generation')
vs     = VectorStore()
router = QueryRouter(llm=llm)

# Load graph + communities
cd = CommunityDetector()
cd.load_graph()
cd.load_partitions()
cs = CommunitySummarizer(detector=cd, llm=llm)

# Graph traversal
gt = GraphTraversal(llm=llm)
gt.load_graph()

# Re-index vector store if empty
if vs.stats()['total_chunks'] == 0:
    print('Re-indexing chunks...')
    chunks = load_all_chunks()
    vs.build_index(chunks)
print('Vector store chunks:', vs.stats()['total_chunks'])

# Load community summaries
cs.summarize_all(resolution=1.0)

graph_engine = GraphQueryEngine(summarizer=cs, llm=llm)

engine = QueryEngine(
    vector_store=vs,
    graph_engine=graph_engine,
    graph_traversal=gt,
    router=router,
    llm=llm,
)

print(f'Graph: {cd.graph.number_of_nodes()} nodes, {cd.graph.number_of_edges()} edges')
members_1 = cd.get_community_members(1.0)
print(f'Communities: {len(members_1)}')
print('\n✅ Cell 2 passed — all engines ready')

In [ ]:
# ── Cell 3: Generate 125 evaluation questions ─────────────────
# Persona-based generation: K=5 personas, N=5 tasks, M=5 questions = 125
# All questions are about the Transformer / Attention paper

CORPUS_DESCRIPTION = """
The paper 'Attention is All You Need' (Vaswani et al., 2017) introduces the
Transformer architecture for sequence-to-sequence tasks. Key topics include:
- The Transformer model architecture (encoder-decoder structure)
- Self-attention and multi-head attention mechanisms
- Scaled dot-product attention
- Positional encodings (sinusoidal and learned)
- Position-wise feed-forward networks
- Layer normalization and residual connections
- Comparison with RNNs, LSTMs, and convolutional sequence models
- Experiments on WMT 2014 English-German and English-French translation
- BLEU score results and training details (Adam optimizer, P100 GPUs)
- Applications to English constituency parsing
"""

PERSONA_GEN_PROMPT = """Given this research paper:
{corpus_desc}

Generate {k} distinct personas who might query a chatbot about this paper.
Include: students, researchers, engineers, and practitioners at different levels.

Return ONLY valid JSON:
[{{"name": "name", "role": "their role", "goal": "what they want to learn"}}]"""

TASK_GEN_PROMPT = """Persona: {persona}
Paper: {corpus_desc}

Generate {n} specific tasks this persona wants to accomplish using this paper.
Return ONLY valid JSON: ["task 1", "task 2", ...]"""

QUESTION_GEN_PROMPT = """Persona: {persona}
Task: {task}
Paper: {corpus_desc}

Generate {m} questions this persona would ask. Mix specific facts AND broad understanding.
Return ONLY valid JSON: ["question 1", "question 2", ...]"""

def parse_json(text):
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        text = '\n'.join(lines[1:-1] if lines[-1].strip()=='```' else lines[1:])
    return json.loads(text)

questions_path = 'evaluation/questions.json'
if os.path.exists(questions_path):
    with open(questions_path) as f:
        all_questions = json.load(f)
    print(f'Loaded {len(all_questions)} questions from cache')
else:
    K, N, M = 5, 5, 5
    all_questions = []

    print('Generating personas...')
    personas = parse_json(llm.generate(
        PERSONA_GEN_PROMPT.format(corpus_desc=CORPUS_DESCRIPTION, k=K)
    ))
    print(f'Personas: {[p["name"] for p in personas]}')
    time.sleep(2)

    for pi, persona in enumerate(personas):
        print(f'\nPersona {pi+1}/{len(personas)}: {persona["name"]}')
        tasks = parse_json(llm.generate(
            TASK_GEN_PROMPT.format(
                persona=json.dumps(persona),
                corpus_desc=CORPUS_DESCRIPTION, n=N
            )
        ))
        time.sleep(1.5)

        for ti, task in enumerate(tasks[:N]):
            questions = parse_json(llm.generate(
                QUESTION_GEN_PROMPT.format(
                    persona=json.dumps(persona),
                    task=task,
                    corpus_desc=CORPUS_DESCRIPTION, m=M
                )
            ))
            time.sleep(1.5)
            for q in questions[:M]:
                all_questions.append({
                    'question_id': f'p{pi+1}_t{ti+1}_q{len(all_questions)+1}',
                    'question':    q,
                    'persona':     persona['name'],
                    'task':        task,
                })
        print(f'  Questions so far: {len(all_questions)}')

    with open(questions_path, 'w') as f:
        json.dump(all_questions, f, indent=2)
    print(f'Saved {len(all_questions)} questions')

print(f'\nTotal: {len(all_questions)} questions')
print('Sample:')
import random
for q in random.sample(all_questions, 5):
    print(f'  [{q["persona"]}] {q["question"]}')
print('\n✅ Cell 3 passed')

In [ ]:
# ── Cell 4: Router accuracy — 30 labeled queries ──────────────
# All queries are about the Attention paper specifically
# Review labels before running — change any you disagree with

labeled_queries = [
    # LOCAL — specific facts from the paper
    {'query': 'What does BLEU stand for and what score did the Transformer achieve on WMT 2014 English-German?', 'label': 'LOCAL'},
    {'query': 'How many attention heads does the base Transformer model use?',                                    'label': 'LOCAL'},
    {'query': 'What optimizer was used to train the Transformer?',                                               'label': 'LOCAL'},
    {'query': 'What is the formula for Scaled Dot-Product Attention?',                                          'label': 'LOCAL'},
    {'query': 'How many encoder layers does the Transformer have?',                                             'label': 'LOCAL'},
    {'query': 'What type of positional encoding does the Transformer use?',                                     'label': 'LOCAL'},
    {'query': 'What GPU was used to train the Transformer model?',                                              'label': 'LOCAL'},
    {'query': 'What is label smoothing and what value was used in Transformer training?',                       'label': 'LOCAL'},
    {'query': 'What is the model dimension d_model in the base Transformer?',                                   'label': 'LOCAL'},
    {'query': 'What tasks were used to evaluate the Transformer beyond translation?',                           'label': 'LOCAL'},

    # GLOBAL — themes and synthesis across the whole paper
    {'query': 'What are the main contributions of the Attention is All You Need paper?',                        'label': 'GLOBAL'},
    {'query': 'How does the Transformer compare to previous sequence-to-sequence models overall?',              'label': 'GLOBAL'},
    {'query': 'What are the key advantages of attention over recurrence according to this paper?',              'label': 'GLOBAL'},
    {'query': 'What are all the components that make up the Transformer architecture?',                         'label': 'GLOBAL'},
    {'query': 'Summarize the experimental results presented in the paper',                                      'label': 'GLOBAL'},
    {'query': 'What future research directions does the paper suggest?',                                        'label': 'GLOBAL'},
    {'query': 'What are the limitations of the Transformer mentioned in the paper?',                           'label': 'GLOBAL'},
    {'query': 'How does the paper motivate moving away from recurrent neural networks?',                        'label': 'GLOBAL'},
    {'query': 'What design choices distinguish the Transformer from all prior architectures?',                  'label': 'GLOBAL'},
    {'query': 'What is the overall narrative and significance of this paper to the NLP field?',                 'label': 'GLOBAL'},

    # HYBRID — need both specific facts and broader understanding
    {'query': 'How does Multi-Head Attention work and why is it better than single attention?',                 'label': 'HYBRID'},
    {'query': 'Explain Scaled Dot-Product Attention and how it fits into the Transformer',                      'label': 'HYBRID'},
    {'query': 'How does positional encoding work and why is it necessary in the Transformer?',                  'label': 'HYBRID'},
    {'query': 'What role do residual connections and layer normalization play in the Transformer?',             'label': 'HYBRID'},
    {'query': 'How does the encoder-decoder attention mechanism connect the two stacks?',                       'label': 'HYBRID'},
    {'query': 'Compare self-attention to recurrent layers in terms of computational complexity',                'label': 'HYBRID'},
    {'query': 'How does the Transformer handle variable length sequences without recurrence?',                  'label': 'HYBRID'},
    {'query': 'Explain the role of the feed-forward network in each Transformer layer',                        'label': 'HYBRID'},
    {'query': 'How does dropout regularization help training and where is it applied in the Transformer?',     'label': 'HYBRID'},
    {'query': 'What is beam search and how was it used in Transformer inference?',                             'label': 'HYBRID'},
]

from collections import Counter
print(f'Labeled queries: {len(labeled_queries)}')
print(f'Distribution: {dict(Counter(q["label"] for q in labeled_queries))}')

print('\nRunning router on all 30 queries...')
correct = 0
results = []

for item in labeled_queries:
    decision = router.route(item['query'])
    predicted  = decision.route_type.value
    is_correct = predicted == item['label']
    if is_correct: correct += 1
    results.append({
        'query':      item['query'],
        'label':      item['label'],
        'predicted':  predicted,
        'correct':    is_correct,
        'confidence': decision.confidence,
    })
    time.sleep(0.5)

accuracy = correct / len(labeled_queries)
print(f'\nRouter accuracy: {correct}/{len(labeled_queries)} = {accuracy:.1%}')

for label in ['LOCAL', 'GLOBAL', 'HYBRID']:
    subset = [r for r in results if r['label'] == label]
    acc    = sum(r['correct'] for r in subset) / len(subset)
    print(f'  {label:8s}: {sum(r["correct"] for r in subset)}/{len(subset)} = {acc:.1%}')

mistakes = [r for r in results if not r['correct']]
if mistakes:
    print(f'\nMisclassified ({len(mistakes)}):')
    for m in mistakes:
        print(f'  Expected {m["label"]} → Got {m["predicted"]}: {m["query"][:65]}')

with open('evaluation/router_accuracy.json', 'w') as f:
    json.dump({'accuracy': accuracy, 'results': results}, f, indent=2)

print('\n✅ Cell 4 passed — router accuracy measured')

In [ ]:
# ── Cell 5: Run all 3 systems on all 125 questions ────────────
# Systems:
#   vanilla_rag → force LOCAL  (vector retrieval only)
#   graphrag    → force GLOBAL (map-reduce only)
#   hybrid      → router decides (full system)
#
# ~375 API calls total. Safe to interrupt — cached after every question.

SYSTEMS = {
    'vanilla_rag': 'LOCAL',
    'graphrag':    'GLOBAL',
    'hybrid':      None,
}

answers_path = 'evaluation/answers.json'
if os.path.exists(answers_path):
    with open(answers_path) as f:
        all_answers = json.load(f)
    print(f'Loaded {len(all_answers)} cached answers')
else:
    all_answers = {}

total_needed = len(all_questions) * len(SYSTEMS)
print(f'Progress: {len(all_answers)}/{total_needed}')
print('Running... (safe to interrupt, resumes from where it stopped)\n')

for qi, q_item in enumerate(all_questions):
    qid = q_item['question_id']

    for system_name, force_route in SYSTEMS.items():
        key = f'{qid}__{system_name}'
        if key in all_answers:
            continue
        try:
            result = engine.query(q_item['question'], force_route=force_route)
            all_answers[key] = {
                'question_id': qid,
                'question':    q_item['question'],
                'system':      system_name,
                'answer':      result['answer'],
                'route':       result['route'],
            }
            time.sleep(1.5)
        except Exception as e:
            print(f'Failed {key}: {e}')
            time.sleep(3.0)

    if qi % 5 == 0:
        with open(answers_path, 'w') as f:
            json.dump(all_answers, f, indent=2)
        print(f'  {len(all_answers)}/{total_needed} — q{qi+1}/{len(all_questions)}')

with open(answers_path, 'w') as f:
    json.dump(all_answers, f, indent=2)

print(f'\nDone. Total answers: {len(all_answers)}')
print('\n✅ Cell 5 passed')

In [ ]:
# ── Cell 6: LLM-as-judge pairwise evaluation ──────────────────
# Compares: vanilla_rag vs hybrid, graphrag vs hybrid
# 3 criteria: comprehensiveness, diversity, directness
# Each pair repeated 3 times

JUDGE_PROMPT = """You are an expert evaluator comparing two AI responses about
the Transformer architecture paper 'Attention is All You Need'.

Question: {question}

Response A:
{response_a}

Response B:
{response_b}

For each criterion choose A, B, or TIE:
1. COMPREHENSIVENESS: Which covers more aspects of the question?
2. DIVERSITY: Which provides more varied perspectives and insights?
3. DIRECTNESS: Which is clearer and more directly addresses the question?

Return ONLY valid JSON:
{{"comprehensiveness": "A"|"B"|"TIE", "diversity": "A"|"B"|"TIE",
  "directness": "A"|"B"|"TIE", "reasoning": "one sentence"}}"""

COMPARISONS  = [('vanilla_rag', 'hybrid'), ('graphrag', 'hybrid')]
JUDGE_REPEATS = 3

judgments_path = 'evaluation/judgments.json'
if os.path.exists(judgments_path):
    with open(judgments_path) as f:
        all_judgments = json.load(f)
    print(f'Loaded {len(all_judgments)} cached judgments')
else:
    all_judgments = {}

total_needed = len(all_questions) * len(COMPARISONS) * JUDGE_REPEATS
print(f'Total judgments needed: {total_needed}')
print(f'Already done: {len(all_judgments)}')
print('Running... (safe to interrupt)\n')

def parse_json(text):
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        text = '\n'.join(lines[1:-1] if lines[-1].strip()=='```' else lines[1:])
    return json.loads(text)

for qi, q_item in enumerate(all_questions):
    qid = q_item['question_id']
    for sys_a, sys_b in COMPARISONS:
        key_a = f'{qid}__{sys_a}'
        key_b = f'{qid}__{sys_b}'
        if key_a not in all_answers or key_b not in all_answers:
            continue
        ans_a = all_answers[key_a]['answer']
        ans_b = all_answers[key_b]['answer']
        for repeat in range(JUDGE_REPEATS):
            j_key = f'{qid}__{sys_a}_vs_{sys_b}__r{repeat}'
            if j_key in all_judgments:
                continue
            try:
                raw = llm.generate(
                    JUDGE_PROMPT.format(
                        question=q_item['question'],
                        response_a=ans_a[:1500],
                        response_b=ans_b[:1500],
                    ), temperature=0.3
                )
                judgment = parse_json(raw)
                judgment.update({'question_id': qid, 'system_a': sys_a,
                                 'system_b': sys_b, 'repeat': repeat})
                all_judgments[j_key] = judgment
                time.sleep(1.0)
            except Exception as e:
                print(f'Failed {j_key}: {e}')
                time.sleep(3.0)

    if qi % 10 == 0:
        with open(judgments_path, 'w') as f:
            json.dump(all_judgments, f, indent=2)
        print(f'  {len(all_judgments)}/{total_needed} — q{qi+1}')

with open(judgments_path, 'w') as f:
    json.dump(all_judgments, f, indent=2)
print(f'\nAll judgments saved: {len(all_judgments)}')
print('\n✅ Cell 6 passed')

In [ ]:
# ── Cell 7: Compute win rates ──────────────────────────────────

def compute_win_rates(judgments, sys_a, sys_b):
    metrics = ['comprehensiveness', 'diversity', 'directness']
    results = {m: {'A_wins': 0, 'B_wins': 0, 'ties': 0} for m in metrics}
    for j in judgments.values():
        if j['system_a'] != sys_a or j['system_b'] != sys_b:
            continue
        for m in metrics:
            v = j.get(m, 'TIE')
            if v == 'A':   results[m]['A_wins'] += 1
            elif v == 'B': results[m]['B_wins'] += 1
            else:          results[m]['ties']   += 1
    win_rates = {}
    for m in metrics:
        total = sum(results[m].values())
        if total == 0:
            win_rates[m] = {f'{sys_a}_win%': 0, f'{sys_b}_win%': 0, 'tie%': 0}
        else:
            win_rates[m] = {
                f'{sys_a}_win%': round(results[m]['A_wins']/total*100, 1),
                f'{sys_b}_win%': round(results[m]['B_wins']/total*100, 1),
                'tie%':         round(results[m]['ties']/total*100, 1),
            }
    return win_rates

with open('evaluation/router_accuracy.json') as f:
    router_data = json.load(f)

wr1 = compute_win_rates(all_judgments, 'vanilla_rag', 'hybrid')
wr2 = compute_win_rates(all_judgments, 'graphrag',    'hybrid')

final_results = {
    'router_accuracy':    router_data['accuracy'],
    'vanilla_vs_hybrid':  wr1,
    'graphrag_vs_hybrid': wr2,
    'total_questions':    len(all_questions),
    'total_judgments':    len(all_judgments),
}
with open('evaluation/final_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print('='*60)
print('EVALUATION RESULTS')
print('='*60)
print(f'Router accuracy: {router_data["accuracy"]:.1%}')

ra = router_data['results']
for label in ['LOCAL', 'GLOBAL', 'HYBRID']:
    subset = [r for r in ra if r['label'] == label]
    acc    = sum(r['correct'] for r in subset) / len(subset)
    print(f'  {label:8s}: {acc:.1%}')

print('\n--- Vanilla RAG vs Hybrid ---')
for metric, rates in wr1.items():
    vals = list(rates.values())
    print(f'  {metric:20s}  vanilla={vals[0]}%  hybrid={vals[1]}%  tie={vals[2]}%')

print('\n--- GraphRAG vs Hybrid ---')
for metric, rates in wr2.items():
    vals = list(rates.values())
    print(f'  {metric:20s}  graphrag={vals[0]}%  hybrid={vals[1]}%  tie={vals[2]}%')

print('\n✅ Cell 7 passed — results computed')

In [ ]:
# ── Cell 8: Clean results table for report ────────────────────

print('RESULTS — paste directly into report')
print('='*65)

print(f'''
Table 1: Router Classification Accuracy
Corpus: Attention is All You Need (Vaswani et al., 2017)
----------------------------------------
Overall: {router_data["accuracy"]:.1%}
''')
ra = router_data['results']
for label in ['LOCAL', 'GLOBAL', 'HYBRID']:
    subset = [r for r in ra if r['label'] == label]
    acc    = sum(r['correct'] for r in subset) / len(subset)
    print(f'  {label:8s}: {acc:.1%}  ({sum(r["correct"] for r in subset)}/{len(subset)})')

print(f'''
Table 2: Win Rates — Hybrid GraphRAG vs Vanilla RAG
Metric                Vanilla wins   Hybrid wins   Tie
''')
for metric, rates in wr1.items():
    v = list(rates.values())
    print(f'  {metric:20s}  {v[0]:>5.1f}%        {v[1]:>5.1f}%      {v[2]:>5.1f}%')

print(f'''
Table 3: Win Rates — Hybrid GraphRAG vs Graph-only
Metric                GraphRAG wins  Hybrid wins   Tie
''')
for metric, rates in wr2.items():
    v = list(rates.values())
    print(f'  {metric:20s}  {v[0]:>5.1f}%        {v[1]:>5.1f}%      {v[2]:>5.1f}%')

print(f'''
Evaluation summary:
  Total questions : {len(all_questions)}
  Total judgments : {len(all_judgments)}
  Judge repeats   : 3x per pair
  Corpus          : Attention is All You Need (1 paper)
''')
print('✅ Evaluation complete — results ready for report')